In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

# =====================================================================
# Pure PyTorch Graph Convolutional Network (GCN) from Scratch
# =====================================================================
# GNNs operate on graphs by passing messages between neighboring nodes.
# Kipf & Welling's Graph Convolutional layer updates node features using:
#     Z = Activation( D^{-1/2} * A_tilde * D^{-1/2} * X * W )
# Where:
#   - A_tilde is the Adjacency matrix + Identity matrix (self-connections)
#   - D is the Degree matrix of A_tilde
#   - X is the input node features
#   - W is the trainable weight matrix
# =====================================================================

# 1. Graph Data Setup
# We build a small graph with 6 nodes (0 to 5) divided into 2 communities:
# Community A: [0, 1, 2] | Community B: [3, 4, 5]
# Connected by a bridge edge: 1-3
num_nodes = 6
edges = [
    (0, 1), (0, 2), (1, 2),  # Community A edges
    (3, 4), (3, 5), (4, 5),  # Community B edges
    (1, 3)                   # Bridge connection
]

# True community labels
labels = torch.tensor([0, 0, 0, 1, 1, 1], dtype=torch.long)

# Semi-supervised training masks (we only train on nodes 0 and 5)
train_mask = torch.tensor([True, False, False, False, False, True], dtype=torch.bool)
test_mask = ~train_mask

# Simple node feature matrix (6 nodes, 4 features each)
features = torch.tensor([
    [1.0, 0.0, 1.0, 0.1],  # Node 0
    [1.1, 0.1, 0.9, 0.0],  # Node 1
    [0.9, 0.0, 1.2, 0.2],  # Node 2
    [0.1, 1.0, 0.0, 0.9],  # Node 3
    [0.0, 1.2, 0.1, 1.1],  # Node 4
    [0.2, 0.9, 0.0, 1.0],  # Node 5
], dtype=torch.float)

# 2. Build and Normalize Adjacency Matrix
print("Building normalized adjacency matrix...")
# Self-connection adjacency matrix A_tilde
adj = torch.eye(num_nodes)  # Start with identity matrix I for self-connections
for u, v in edges:
    adj[u, v] = 1.0
    adj[v, u] = 1.0  # Undirected graph

# Calculate degrees: row-sums of adj
degrees = torch.sum(adj, dim=1)

# Compute symmetric inverse-square-root degree matrix: D^{-1/2}
deg_inv_sqrt = torch.diag(1.0 / torch.sqrt(degrees))

# Compute normalized adjacency: A_norm = D^{-1/2} * A_tilde * D^{-1/2}
adj_norm = torch.mm(torch.mm(deg_inv_sqrt, adj), deg_inv_sqrt)

# 3. GCN Layer from Scratch
class GCNLayer(nn.Module):
    def __init__(self, in_features, out_features):
        super(GCNLayer, self).__init__()
        # Trainable weight projection
        self.projection = nn.Linear(in_features, out_features, bias=False)

    def forward(self, x, adj_norm):
        # Step A: Project node features: X * W
        x_projected = self.projection(x)
        # Step B: Aggregate neighbor representations: A_norm * (X * W)
        x_aggregated = torch.mm(adj_norm, x_projected)
        return x_aggregated

# 4. Two-Layer GCN Network
class GCN(nn.Module):
    def __init__(self, in_features, hidden_dim, num_classes):
        super(GCN, self).__init__()
        self.conv1 = GCNLayer(in_features, hidden_dim)
        self.relu = nn.ReLU()
        self.conv2 = GCNLayer(hidden_dim, num_classes)

    def forward(self, x, adj_norm):
        h = self.conv1(x, adj_norm)
        h = self.relu(h)
        h = self.conv2(h, adj_norm)
        return h

# Initialize model, loss and optimizer
# Predict 2 classes (Community A vs Community B)
model = GCN(in_features=4, hidden_dim=4, num_classes=2)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)

# 5. Training Loop (Semi-supervised Node Classification)
print("Training Graph Convolutional Network...")
model.train()
for epoch in range(100):
    optimizer.zero_grad()
    # Forward pass on the ENTIRE graph
    out = model(features, adj_norm)
    # Calculate loss ONLY on the training nodes (0 and 5)
    loss = criterion(out[train_mask], labels[train_mask])
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 20 == 0:
        print(f"Epoch {epoch+1:03d}/100 | Train Loss: {loss.item():.4f}")

# 6. Evaluation
model.eval()
with torch.no_grad():
    logits = model(features, adj_norm)
    predictions = torch.argmax(logits, dim=1)

    print("\n--- Node Classification Results ---")
    for i in range(num_nodes):
        node_type = "Train" if train_mask[i] else "Test"
        correct = "CORRECT" if predictions[i] == labels[i] else "WRONG"
        print(f"Node {i} ({node_type}) | Predicted Class: {predictions[i].item()} | True Class: {labels[i].item()} | {correct}")


Building normalized adjacency matrix...
Training Graph Convolutional Network...
Epoch 020/100 | Train Loss: 0.5196
Epoch 040/100 | Train Loss: 0.4013
Epoch 060/100 | Train Loss: 0.3618
Epoch 080/100 | Train Loss: 0.3113
Epoch 100/100 | Train Loss: 0.1059

--- Node Classification Results ---
Node 0 (Train) | Predicted Class: 0 | True Class: 0 | CORRECT
Node 1 (Test) | Predicted Class: 0 | True Class: 0 | CORRECT
Node 2 (Test) | Predicted Class: 0 | True Class: 0 | CORRECT
Node 3 (Test) | Predicted Class: 1 | True Class: 1 | CORRECT
Node 4 (Test) | Predicted Class: 1 | True Class: 1 | CORRECT
Node 5 (Train) | Predicted Class: 1 | True Class: 1 | CORRECT
